# Extracting Narratives

#### INPUT  
- preprocessed_dataset.tsv

#### WHAT THIS NOTEBOOK DOES
- loads the .tsv file preprocessed with notebook 02_preprocess.ipynb
- splits the data into batches
- uses the Anthropic batch API to extract narrative elements as defined in prompts/extract_prompt
- tracks progress across runs so extraction can be resumed, and retries failed batches
- writes the extracted data into new files

#### OUTPUTS 
- narrative_extractions.csv --> extracted narrative elements in long format (one row per slot filler)
- narrative_reasoning.csv --> model reasoning steps
- extractions_raw.jsonl --> raw model responses (intermediate)
- extraction_state.json / batch_ids.json / batch_ids_retry.json --> batch tracking / resume state


In [ ]:
from config import extract_config as cfg
from lib.extract_lib import (
    make_user_message,
    load_state,
    dispatch_all, poll_until_complete, collect_results,
    parse_and_validate, retry_failed,
    print_sanity_checks,
    save_reasoning, to_long_format,
)
from prompts.extract_prompt import SYSTEM_PROMPT

import anthropic
import pandas as pd
import json

client = anthropic.Anthropic()


In [2]:
# load the preprocessed dataframe

df = pd.read_csv(cfg.INPUT_TSV, sep="\t")
df["id"] = df["id"].astype(str)
df = df.drop_duplicates(subset="id").reset_index(drop=True)

print(f"Articles loaded: {len(df):,}")
df.head()

Articles loaded: 64,504


,id,pubtime,medium_code,medium_name,rubric,char_count,head,content,publisher,sentiment_score,emotion_anger,emotion_joy,emotion_sadness,emotion_fear,linguistic_complexity
0,59217348,2025-12-16 00:00:00+01,NZZ,Neue Zürcher Zeitung,International,7943,Die Rechte erlebt in Chile eine Renaissance,José Antonio Kast hat Bewunderung für Pinochet...,NZZ-Mediengruppe,-0.035220,0.712058,0.085402,0.178484,0.024056,52.567486
1,60051005,2026-02-23 00:00:00+01,BZ,Berner Zeitung,Ausland,775,Bei Trump-Anwesen: Sicherheitskräfte erschiess...,USA In der Nacht auf gestern ist eine Person u...,TX Group,-0.823971,0.624077,0.036465,0.154683,0.184774,54.405636
2,58712744,2025-11-01 08:55:22+01,BLIO,blick.ch,Wirtschaft,5185,Immer mehr GaultMillau-Köche geben auf: Was is...,Die gehobene Gastronomie in der Schweiz erlebt...,Ringier,-0.045350,0.418795,0.177447,0.352456,0.051302,50.120130
3,58259935,2025-09-16 16:57:12+02,BLIO,blick.ch,Schweiz,989,Halbe Million Kubikmeter Gestein: Steinschlag ...,Am Wochenende rumorte es im Unterengadin. In d...,Ringier,-0.942938,0.044182,0.010790,0.045057,0.899971,56.704830
4,59618286,2026-01-30 22:14:03+01,BLIO,blick.ch,Ausland,740,Über 113 Millionen: Glückspilz knackt Euro-Mil...,Ein Glückspilz hat am Freitag bei Euro Million...,Ringier,-0.003302,0.017826,0.956364,0.018452,0.007358,57.207589


In [10]:
# extract narrative from one sample article using the prompts in prompts/extract_prompt.py
sample_row = df.sample(1, random_state=42).iloc[0]
response = client.messages.create(
    model=cfg.MODEL,
    max_tokens=cfg.MAX_TOKENS,
    temperature=cfg.TEMPERATURE,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": make_user_message(sample_row["head"], sample_row["content"])}],
)
print(response.content[0].text)


I'll work through each step carefully.

## STEP 0 — NARRATIVE ASSESSMENT

The article describes a deliberate act (an attack on National Guard soldiers) and its political consequences, with Trump actively pursuing a harder immigration policy in response. There is an identifiable agent (Trump/his administration) pursuing an identifiable goal (stricter immigration enforcement, justification of military deployment in DC). Multiple other slots are fillable. Both tests pass.

**narrative_present: true**

## STEP 1 — DOMINANT NARRATIVE

Trump and his administration seize on the attack by an Afghan suspect to justify and intensify restrictive immigration policy and the controversial domestic military deployment in Washington DC, against legal and political opposition.

## STEP 2 & 3 — ACTANTIAL SLOTS

**Subject:** Trump / the Trump administration — the active agent pursuing stricter immigration policy and defending the National Guard deployment. Foreign_actor / foreign_executive (US government

In [18]:
# send articles to LLM API for extraction

PILOT = False        # False for the full run with all data, True for a quicker pilot run with a small random sample
N_PILOT = 500

state = load_state()

if PILOT:
    pilot_df  = df.sample(N_PILOT, random_state=11)
    batch_ids = dispatch_all(client, pilot_df, state)
else:
    batch_ids = dispatch_all(client, df, state)

# persist batch_ids so a kernel restart doesn't lose them
with open("data/batch_ids.json", "w") as f:
    json.dump(batch_ids, f)

print(f"Dispatched {len(batch_ids)} batch(es): {batch_ids}")

Dispatching batches:  14%|█▍        | 1/7 [00:55<05:33, 55.54s/it]

  Dispatched msgbatch_011ELaSSqL1nCZ58h8nE4ASE (10,000 articles)


Dispatching batches:  29%|██▊       | 2/7 [01:48<04:30, 54.05s/it]

  Dispatched msgbatch_0196F1LhPjdXkMMXD7J8He2m (10,000 articles)


Dispatching batches:  43%|████▎     | 3/7 [02:40<03:31, 52.99s/it]

...

  Dispatched msgbatch_01AqaWFW9qSLR7snTEc7e8CT (10,000 articles)


Dispatching batches:  86%|████████▌ | 6/7 [05:28<00:56, 56.15s/it]

  Dispatched msgbatch_01XTZByA97Ytdvc7ZyrWx51G (10,000 articles)


Dispatching batches: 100%|██████████| 7/7 [06:12<00:00, 53.22s/it]

  Dispatched msgbatch_012pcGERFmeaaabrAxrTLRM2 (3,954 articles)
Dispatched 7 batch(es): ['msgbatch_011ELaSSqL1nCZ58h8nE4ASE', 'msgbatch_0196F1LhPjdXkMMXD7J8He2m', 'msgbatch_019Z7rz6nRNKa683CLbk3dH4', 'msgbatch_01P7snmV6pW4d3Gig8FuHZBr', 'msgbatch_01AqaWFW9qSLR7snTEc7e8CT', 'msgbatch_01XTZByA97Ytdvc7ZyrWx51G', 'msgbatch_012pcGERFmeaaabrAxrTLRM2']


In [19]:
# check on the status of the batches and collect results when complete

with open("data/batch_ids.json") as f:
    batch_ids = json.load(f)
state = load_state()

poll_until_complete(client, batch_ids) # blocks until all batches are complete, updating state in-place as it goes
collect_results(client, batch_ids, state)

  msgbatch_012pcGERFmeaaabrAxrTLRM2: in_progress — processing=3954 succeeded=0 errored=0
  msgbatch_019Z7rz6nRNKa683CLbk3dH4: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01AqaWFW9qSLR7snTEc7e8CT: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01XTZByA97Ytdvc7ZyrWx51G: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_011ELaSSqL1nCZ58h8nE4ASE: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_0196F1LhPjdXkMMXD7J8He2m: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01P7snmV6pW4d3Gig8FuHZBr: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_012pcGERFmeaaabrAxrTLRM2: in_progress — processing=3954 succeeded=0 errored=0
  msgbatch_019Z7rz6nRNKa683CLbk3dH4: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01XTZByA97Ytdvc7ZyrWx51G: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01AqaWFW9qSLR7snTEc7e8CT: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_0

In [20]:
# parse the raw JSONL output and validate against the output schema, separating out malformed JSON and schema validation failures for review

records, malformed, invalid = parse_and_validate(cfg.RAW_OUTPUT)
print(f"Valid:          {len(records):,}") # valid records (model produced parseable JSON that also passed schema validation)
print(f"Malformed JSON: {len(malformed):,}") # JSON could not be extracted at all (model produced no parseable JSON)
print(f"Schema invalid: {len(invalid):,}") # JSON was extracted but failed schema validation (model produced JSON, but with errors in slot filling or formatting)

Valid:          47,762
Malformed JSON: 7
Schema invalid: 1,161


In [21]:
# retry failures (malformed JSON or schema validation failures) and collect results again
# records, malformed, invalid = parse_and_validate(cfg.RAW_OUTPUT) # makes cell independent so you can run it again after fixing issues in the raw output file

state = load_state()
retry_ids = retry_failed(client, df, state, malformed, invalid)

if retry_ids:
    with open("data/batch_ids_retry.json", "w") as f:
        json.dump(retry_ids, f)

    poll_until_complete(client, retry_ids)
    collect_results(client, retry_ids, state)
    records, malformed, invalid = parse_and_validate(cfg.RAW_OUTPUT)
    print(f"After retry — Valid: {len(records):,}  Malformed: {len(malformed):,}  Invalid: {len(invalid):,}")
else:
    print("Nothing to retry.")

Dispatching batches:  50%|█████     | 1/2 [00:52<00:52, 52.58s/it]

  Dispatched msgbatch_01LMeScrQd4LsrD4cM36buKg (10,000 articles)


Dispatching batches: 100%|██████████| 2/2 [01:27<00:00, 43.87s/it]

  Dispatched msgbatch_01CDujnWLJdANHMMS6Ufj5xn (6,745 articles)


  msgbatch_01CDujnWLJdANHMMS6Ufj5xn: in_progress — processing=6745 succeeded=0 errored=0
  msgbatch_01LMeScrQd4LsrD4cM36buKg: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01CDujnWLJdANHMMS6Ufj5xn: in_progress — processing=6745 succeeded=0 errored=0
  msgbatch_01LMeScrQd4LsrD4cM36buKg: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01CDujnWLJdANHMMS6Ufj5xn: in_progress — processing=6745 succeeded=0 errored=0
  msgbatch_01LMeScrQd4LsrD4cM36buKg: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01CDujnWLJdANHMMS6Ufj5xn: in_progress — processing=6745 succeeded=0 errored=0
  msgbatch_01LMeScrQd4LsrD4cM36buKg: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01CDujnWLJdANHMMS6Ufj5xn: in_progress — processing=6745 succeeded=0 errored=0
  msgbatch_01LMeScrQd4LsrD4cM36buKg: in_progress — processing=10000 succeeded=0 errored=0
  msgbatch_01CDujnWLJdANHMMS6Ufj5xn: in_progress — processing=6745 succeeded=0 errored=0
  msgbatch_01LMe

In [22]:
#sanity checks on the extracted records
records, malformed, invalid = parse_and_validate(cfg.RAW_OUTPUT) #makes cell independent so you can run it again after fixing issues in the raw output file

print_sanity_checks(records)

if invalid:
    print(f"\n=== Schema validation errors (first 5) ===")
    for r in invalid[:5]:
        print(f"  id={r['id']}  error={r['error']}")

=== Narrative presence ===
  Narrative present:  44,876 (70.6%)
  No narrative:       18,716 (29.4%)

=== Entity type distribution by slot ===
  subject     actor=100%  abstract_force=0%  ideology=0%  (n=54,491)
  object      actor=0%  abstract_force=81%  ideology=19%  (n=54,567)
  sender      actor=6%  abstract_force=16%  ideology=78%  (n=64,946)
  receiver    actor=98%  abstract_force=2%  ideology=0%  (n=64,922)
  helper      actor=61%  abstract_force=39%  ideology=0%  (n=78,284)
  opponent    actor=50%  abstract_force=45%  ideology=5%  (n=100,373)

=== Empty slot frequency ===
  helper    : 6,825 (15.2%)
  opponent  : 1,136 (2.5%)
  sender    : 209 (0.5%)
  object    : 8 (0.0%)
  receiver  : 4 (0.0%)

=== Uncertain flag usage ===
  Articles with ≥1 uncertain annotation : 32,400 (72.2%)
  Mean uncertain annotations per article: 1.16

=== Schema validation errors (first 5) ===
  id=60463507  error='general_public' is not one of ['actor', 'abstract_force', 'ideology']
  id=59423101  er

In [23]:
# save the output in long format for analysis and clustering

records, _, _ = parse_and_validate(cfg.RAW_OUTPUT)

results_df = to_long_format(records, df)
results_df.to_csv(cfg.OUTPUT_CSV, index=False)
print(f"Saved {len(results_df):,} rows ({results_df['id'].nunique():,} articles) → {cfg.OUTPUT_CSV}")

save_reasoning(records, df) # currently saves an empty CSV since the model is prompted to output only the JSON

results_df.head()

Saved 417,583 rows (44,876 articles) → data/narrative_extractions.csv
Saved 44,876 reasoning texts -> data/narrative_reasoning.csv


,id,narrative_present,dominant_narrative,slot,filler_rank,label,filler_type,specific_type,category,domain,uncertain,uncertainty_note,pubtime,medium_code,medium_name,char_count
0,58672106,True,Russian Foreign Minister Lavrov pursues intern...,subject,0,Sergei Lavrov / Russian government,actor,foreign_executive,foreign_actor,international_diplomatic,False,NaN,2025-10-28 21:27:22+01,ZWAO,20 minuten online,2744
1,58672106,True,Russian Foreign Minister Lavrov pursues intern...,object,0,International acceptance of Russia as security...,abstract_force,policy_instrument,deployed_instrument,international_security,False,NaN,2025-10-28 21:27:22+01,ZWAO,20 minuten online,2744
2,58672106,True,Russian Foreign Minister Lavrov pursues intern...,sender,0,Russian state interest in restoring geopolitic...,ideology,communitarian,cultural_axis,international_diplomatic,True,Lavrov's framing appeals to sovereignty and re...,2025-10-28 21:27:22+01,ZWAO,20 minuten online,2744
3,58672106,True,Russian Foreign Minister Lavrov pursues intern...,receiver,0,EU and NATO member states,actor,foreign_executive,foreign_actor,international_security,False,NaN,2025-10-28 21:27:22+01,ZWAO,20 minuten online,2744
4,58672106,True,Russian Foreign Minister Lavrov pursues intern...,opponent,0,Russia's documented history of treaty violatio...,abstract_force,geopolitical,structural_condition,international_security,False,NaN,2025-10-28 21:27:22+01,ZWAO,20 minuten online,2744
